# Bloom 2.0 — YOLOv8 Training (Colab)

One cell per step. Run top to bottom.

**Before starting:** Runtime → Change runtime type → **GPU (T4)**.

This trains the full pipeline on the harmonized dataset. For the 4 augmentation
ablation runs, change `AUG_MODE` in step 4 and re-run steps 4–6 for each of:
`none`, `geometric`, `photometric`, `full`.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics==8.2.18 datasets==2.19.1 albumentations==1.4.6 \
    torchmetrics==1.4.0 scikit-learn==1.5.0 pandas==2.2.2 Pillow==10.3.0 \
    imagehash==4.3.1 pyarrow==16.1.0 tqdm==4.66.4 pyyaml==6.0.1 opencv-python-headless

## 2. Get the data + code

Option A (recommended): upload the repo's `ml/` folder to Google Drive, then mount.
Minimum you need on Drive: `ml/datasets/harmonized/` (or re-run fetch+harmonize here)
and the `ml/` code files.

Option B: clone the git repo (if it has the harmonized data via LFS or you re-run
fetch_acne04.py + harmonize.py in this notebook).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at where you put the repo on Drive:
REPO = '/content/drive/MyDrive/bloom'   # <-- EDIT ME
%cd {REPO}

# Sanity check: does the harmonized dataset exist here?
import os
assert os.path.exists('ml/data/harmonized/classes.yaml'), \
    'harmonized dataset not found — run fetch_acne04.py + harmonize.py first'
print('dataset OK')

## 3. (Only if data missing on Drive) fetch + harmonize here

Skip this cell if step 2 found the dataset.

In [ ]:
# !python ml/datasets/fetch_acne04.py
# !python ml/datasets/harmonize.py

## 4. Train

Set `AUG_MODE` for each ablation run. Checkpoints + `run_log.csv` land in `ml/runs/`.

In [ ]:
AUG_MODE = 'full'   # 'none' | 'geometric' | 'photometric' | 'full'

!python ml/train/train_yolo.py --aug-mode {AUG_MODE}

## 5. Copy best.pt to Drive (survives runtime disconnect)

In [ ]:
import glob, os, shutil, time

runs = sorted(glob.glob('ml/runs/*_*/best.pt'), key=os.path.getmtime)
assert runs, 'no best.pt found — did training finish?'
latest = runs[-1]
dest_dir = '/content/drive/MyDrive/bloom_models'   # <-- EDIT ME
os.makedirs(dest_dir, exist_ok=True)
dest = os.path.join(dest_dir, os.path.basename(os.path.dirname(latest)) + '_best.pt')
shutil.copy2(latest, dest)
print('saved:', dest)

## 6. Resume support

If the runtime dies mid-run: ultralytics checkpoints every epoch under
`ml/runs/{run_id}/weights/last.pt`. To resume, re-run step 4 with the same
AUG_MODE — a NEW run_id is created (by design, so the log stays honest).
To literally resume weights, uncomment:

```python
from ultralytics import YOLO
model = YOLO('ml/runs/<run_id>/weights/last.pt')
model.train(resume=True)
```